# LLM PCFG Cracking — Colab Training

**前置條件：** Runtime > Change runtime type → **GPU**

**每次 session 執行順序：** Cell 1 → 2 → 3 → 4 → 5

**監控訓練：** 訓練跑著時，另外執行 Cell 6a + 6b 開 TensorBoard

## 1. Clone Repo

In [3]:
import os

REPO_URL = 'https://github.com/jjwang1118/segment_pw_cracking.git'
REPO_DIR = '/content/llm_pcfg_cracking'

if os.path.isdir(REPO_DIR + '/.git'):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('工作目錄：', os.getcwd())

Cloning into '/content/llm_pcfg_cracking'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 304 (delta 3), reused 8 (delta 2), pack-reused 283 (from 1)
Receiving objects: 100% (304/304), 134.02 MiB | 20.73 MiB/s, done.
Resolving deltas: 100% (123/123), done.
Updating files: 100% (110/110), done.
工作目錄： /content/llm_pcfg_cracking


## 2. 安裝依賴

In [4]:
!pip install -q torch transformers datasets peft tokenizers accelerate bitsandbytes scipy pyyaml pandas numpy
print('done')

done


## 3. 確認 GPU，下載 / 快取 Qwen3-4B

In [5]:
import torch
from pathlib import Path
from google.colab import drive

assert torch.cuda.is_available(), 'No GPU!'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU : {gpu_name}  ({vram_gb:.1f} GB)')

# T4 16GB: train backward 需要連續空間，batch=16 會 OOM，降到 8
if vram_gb >= 35:
    BATCH, EVAL_BATCH, GRAD_ACCUM = 16, 16, 64    # A100 40GB
else:
    BATCH, EVAL_BATCH, GRAD_ACCUM = 8, 8, 128    # T4 16GB
print(f'train_batch={BATCH}, eval_batch={EVAL_BATCH}, grad_accum={GRAD_ACCUM}  → effective {BATCH*GRAD_ACCUM}')

drive.mount('/content/drive')
DRIVE_DIR   = '/content/drive/MyDrive/llm_pcfg_cracking_model'
DRIVE_MODEL = DRIVE_DIR + '/Qwen3-4B'
LOCAL_MODEL = REPO_DIR  + '/models/Qwen3-4B'
os.makedirs(DRIVE_DIR, exist_ok=True)

if Path(DRIVE_MODEL + '/config.json').exists():
    print('從 Drive 快取載入...')
    !ln -sfn {DRIVE_MODEL} {LOCAL_MODEL}
    print('✓ symlink 完成')
else:
    print('首次下載 Qwen/Qwen3-4B（約 8GB）...')
    from transformers import AutoTokenizer, AutoModelForCausalLM
    _tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')
    _tok.save_pretrained(LOCAL_MODEL)
    _mdl = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B', torch_dtype=torch.bfloat16)
    _mdl.save_pretrained(LOCAL_MODEL)
    del _tok, _mdl; torch.cuda.empty_cache()
    !rsync -a --info=progress2 {LOCAL_MODEL}/ {DRIVE_MODEL}/
    print('✓ 完成')

GPU : Tesla T4  (14.6 GB)
train_batch=8, eval_batch=8, grad_accum=128  → effective 1024
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
從 Drive 快取載入...
✓ symlink 完成


## 5. TensorBoard 監控

訓練進行中可執行此 cell，約 10 steps 後才有資料。

In [6]:
# 確認 event file 存在
!find /content/llm_pcfg_cracking/checkpoints -name 'events.out.*' 2>/dev/null

# 背景啟動 TensorBoard
!pkill -f tensorboard 2>/dev/null
!sleep 1
!nohup tensorboard --logdir /content/llm_pcfg_cracking/checkpoints --host 0.0.0.0 --port 6006 > /tmp/tb.log 2>&1 &
!sleep 3 && echo '=== 啟動完成 ===' && cat /tmp/tb.log

^C
=== 啟動完成 ===
2026-06-22 05:25:25.941200: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [7]:
from google.colab import output
output.serve_kernel_port_as_iframe(6006, height=800)

<IPython.core.display.Javascript object>

## 4. 執行訓練

In [ ]:
# （Resume 時執行）從 Drive 還原 checkpoints 到本地
# 首次訓練不需執行此 cell
import subprocess
DRIVE_CKPT = DRIVE_DIR + "/checkpoints"
LOCAL_CKPT = REPO_DIR  + "/checkpoints"
!rsync -a --info=progress2 {DRIVE_CKPT}/ {LOCAL_CKPT}/
print("✓ checkpoint 已從 Drive 還原")
!find {LOCAL_CKPT} -name "trainer_state.json" | sort

In [ ]:
import yaml, os

config_path = Path(REPO_DIR) / 'config' / 'train_config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

tc = cfg['train']['train_config']
tc['per_device_train_batch_size'] = BATCH
tc['per_device_eval_batch_size']  = EVAL_BATCH
tc['gradient_accumulation_steps'] = GRAD_ACCUM
tc['model_name'] = 'Qwen3-4B'
tc['model_path'] = 'models'
tc['dataloader_num_workers'] = 2

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print(f'train_batch={BATCH}, eval_batch={EVAL_BATCH}, grad_accum={GRAD_ACCUM}')
os.environ['TOKENIZERS_PARALLELISM']   = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTHONUNBUFFERED']         = '1'

!python -u run_train.py
# 從 checkpoint 繼續（取消下方註解）：
# !python -u run_train.py --resume checkpoints/Qwen3-4B/run_2/checkpoint-400

train_batch=8, eval_batch=8, grad_accum=128
Generating train split: 276487 examples [00:00, 997592.65 examples/s] 
Generating test split: 7089 examples [00:00, 921737.90 examples/s]
[*] QLoRA 模式：4-bit NF4 量化載入 /content/llm_pcfg_cracking/models/Qwen3-4B
Loading weights:   0% 1/398 [00:12<1:20:21, 12.14s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 398/398 [02:26<00:00,  2.72it/s]
Map: 100% 276487/276487 [03:40<00:00, 1256.43 examples/s]
Map: 100% 7089/7089 [00:05<00:00, 1412.81 examples/s]
[*] 訓練輸出目錄：/content/llm_pcfg_cracking/checkpoints/Qwen3-4B/run_1
[*] QLoRA overrides applied: {'bf16': False, 'optim': 'paged_adamw_8bit'}
[*] total_steps=810  warmup_steps=81
2026-06-22 05:32:11.423652: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow b

## 5. 備份 Checkpoint 至 Drive

In [ ]:
DRIVE_CKPT = DRIVE_DIR + '/checkpoints'
LOCAL_CKPT = REPO_DIR  + '/checkpoints'
!rsync -a --info=progress2 {LOCAL_CKPT}/ {DRIVE_CKPT}/
print('✓ checkpoint 已備份至 Drive')
!find {DRIVE_CKPT} -name 'adapter_config.json' | sort